# ── CELL 1: Install dependencies ────



In [ ]:
%%capture
!pip install unsloth
!pip install datasets
print("Done!")

# ── CELL 2:  config ────

In [ ]:
import os, json, torch
import pandas as pd
from google.colab import drive
from unsloth import FastModel

drive.mount('/content/drive')

MODEL_ID    = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"
TEST_CSV    = "/content/drive/MyDrive/VisualStep/test_set_200.csv"
RESULTS_DIR = "/content/drive/MyDrive/VisualStep/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

SYSTEM_PROMPT = """You are an expert in ADHD classroom accommodations for primary school children aged 5-10.
Transform the teacher's spoken classroom instruction into an ADHD-adapted visual step card.

Rules:
- Maximum 5 steps
- Each step: one concrete physical action only
- Each action: maximum 6 words
- Start each step with an imperative verb
- Include a pictogram keyword in the icon field
- Output JSON only, no other text

Output format:
{"steps":[{"id":1,"action":"Open your book","detail":"page 23","icon":"book","check":true}],"n_steps":1,"support_level":"medium"}

support_level: "light" (1-2 steps), "medium" (3-4 steps), "high" (5 steps)"""

print("Config ready")

# Cell 3 — Load base model (no adapter)

In [ ]:
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=512,
    load_in_4bit=True,
    dtype=None,
)
FastModel.for_inference(model)
model.eval()
print("Base model loaded — no LoRA adapter")

# CELL 4 : Run Zero-shot Phi-3-mini-4k-instruct-bnb-4bit


In [ ]:
import pandas as pd

df_test = pd.read_csv("/content/drive/MyDrive/VisualStep/test_set_200.csv")
print(f"Test set loaded: {len(df_test)} examples")

df_test.to_csv(
    "/content/drive/MyDrive/VisualStep/test_set_200.csv",
    index=False,
    encoding='utf-8'
)
print(f"Saved: {len(df_test)} examples")

def generate_step_card(instruction, year, subject, max_new_tokens=256):
    user_content = f"Year: {year} | Subject: {subject} | Instruction: {instruction}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

results = []
ok = fail = 0

for i, (_, row) in enumerate(df_test.iterrows()):
    if i % 10 == 0:
        print(f"  [{i+1}/{len(df_test)}]", end="", flush=True)

    output = generate_step_card(row['input'], row['year'], row['subject'])

    try:
        parsed     = json.loads(output)
        valid_json = True
        n_steps    = parsed.get('n_steps', len(parsed.get('steps', [])))
        support    = parsed.get('support_level', '')
        ok += 1
    except json.JSONDecodeError:
        valid_json = False
        n_steps    = 0
        support    = ''
        fail += 1

    results.append({
        'id':            row.get('id', f'TEST-{i}'),
        'source':        row.get('source', ''),
        'year':          row['year'],
        'subject':       row['subject'],
        'type':          row.get('type', ''),
        'input':         row['input'],
        'gold_output':   str(row.get('output_json', '')),
        'model_output':  output,
        'valid_json':    valid_json,
        'n_steps':       n_steps,
        'support_level': support,
    })

print(f"\n\nResults: {ok} valid JSON | {fail} invalid")

results_path = os.path.join(RESULTS_DIR, "test_outputs_phi3_zeroshot.csv")
pd.DataFrame(results).to_csv(results_path, index=False, encoding='utf-8')
print(f"Saved: {results_path}")

# Cell 5 — Summary

In [ ]:
df_results = pd.DataFrame(results)

print("="*50)
print("Zero-shot Phi-3 Mini — Test Results")
print("="*50)
print(f"Total examples    : {len(df_results)}")
print(f"Valid JSON        : {df_results['valid_json'].sum()} ({df_results['valid_json'].mean()*100:.1f}%)")
print(f"Steps in [1-5]   : {((df_results['n_steps']>=1) & (df_results['n_steps']<=5)).sum()}")
print(f"Mean steps/card  : {df_results[df_results['valid_json']]['n_steps'].mean():.2f}")
print(f"Support level    : {df_results['support_level'].value_counts().to_dict()}")
print(f"\nSaved: {results_path}")
print("="*50)

# Cell 6: Metrics evaluation for Zero-shot Phi-3 Mini

In [ ]:
!pip install rouge-score sacrebleu -q

import json, pandas as pd
from rouge_score import rouge_scorer
from sacrebleu.metrics import BLEU

def compute_metrics(df_results, model_name):
    def extract_actions(json_str):
        try:
            data = json.loads(json_str)
            return " ".join(
                f"{s.get('action','')} {s.get('detail','')}".strip()
                for s in data.get("steps", [])
            )
        except:
            return ""

    hyps, refs = [], []
    for _, row in df_results.iterrows():
        h = extract_actions(str(row["model_output"]))
        r = extract_actions(str(row["gold_output"]))
        if h and r:
            hyps.append(h)
            refs.append(r)

    scorer = rouge_scorer.RougeScorer(
        ["rouge1","rouge2","rougeL"], use_stemmer=True
    )
    r1 = r2 = rL = 0
    for h, r in zip(hyps, refs):
        s = scorer.score(r, h)
        r1 += s["rouge1"].fmeasure
        r2 += s["rouge2"].fmeasure
        rL += s["rougeL"].fmeasure
    n = len(hyps)
    bleu = BLEU(effective_order=True)
    bleu_score = bleu.corpus_score(hyps, [refs]).score

    print(f"\n── {model_name} ──────────────────────────")
    print(f"Valid pairs : {n}")
    print(f"ROUGE-1     : {r1/n:.4f}")
    print(f"ROUGE-2     : {r2/n:.4f}")
    print(f"ROUGE-L     : {rL/n:.4f}")
    print(f"BLEU-4      : {bleu_score:.4f}")
    return {"model": model_name, "valid_json": len(df_results[df_results['valid_json']==True]),
            "rouge1": r1/n, "rouge2": r2/n, "rougeL": rL/n, "bleu4": bleu_score}

# Run for zero-shot Phi-3 Mini
df_phi3_zs = pd.read_csv(
    "/content/drive/MyDrive/VisualStep/results/test_outputs_phi3_zeroshot.csv"
)
result_phi3_zs = compute_metrics(df_phi3_zs, "Zero-shot Phi-3 Mini")

# Cell 7 — Zero-shot Gemma 2 2B Instruct baseline


In [ ]:
df_test = pd.read_csv("/content/drive/MyDrive/VisualStep/test_set_200.csv")

import torch

def generate_step_card(instruction, year, subject, max_new_tokens=256):
    user_content = f"{SYSTEM_PROMPT}\n\nYear: {year} | Subject: {subject} | Instruction: {instruction}"
    messages = [
        {"role": "user", "content": user_content},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

MODEL_ID = "unsloth/gemma-2-2b-it-bnb-4bit"

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=512,
    load_in_4bit=True,
    dtype=None,
)
FastModel.for_inference(model)
model.eval()
print("Gemma 2 2B loaded")

results = []
ok = fail = 0

for i, (_, row) in enumerate(df_test.iterrows()):
    if i % 10 == 0:
        print(f"  [{i+1}/{len(df_test)}]", end="", flush=True)

    output = generate_step_card(row['input'], row['year'], row['subject'])

    try:
        parsed     = json.loads(output)
        valid_json = True
        n_steps    = parsed.get('n_steps', len(parsed.get('steps', [])))
        support    = parsed.get('support_level', '')
        ok += 1
    except json.JSONDecodeError:
        valid_json = False
        n_steps    = 0
        support    = ''
        fail += 1

    results.append({
        'id':            row.get('id', f'TEST-{i}'),
        'source':        row.get('source', ''),
        'year':          row['year'],
        'subject':       row['subject'],
        'type':          row.get('type', ''),
        'input':         row['input'],
        'gold_output':   str(row.get('output_json', '')),
        'model_output':  output,
        'valid_json':    valid_json,
        'n_steps':       n_steps,
        'support_level': support,
    })

results_path = os.path.join(RESULTS_DIR, "test_outputs_gemma2_2b_zeroshot.csv")
pd.DataFrame(results).to_csv(results_path, index=False, encoding='utf-8')

print(f"\n\nResults: {ok} valid JSON | {fail} invalid")
print(f"Saved: {results_path}")

In [ ]:
import re

def clean_output(text):
    text = text.strip()
    # Remove ```json ... ``` or ``` ... ```
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

# Re-parse the existing CSV without re-running inference
import json

results = []
for _, row in df.iterrows():
    cleaned = clean_output(str(row["model_output"]))
    try:
        parsed = json.loads(cleaned)
        valid_json = True
        n_steps = parsed.get("n_steps", len(parsed.get("steps", [])))
        support = parsed.get("support_level", "")
    except json.JSONDecodeError:
        valid_json = False
        n_steps = 0
        support = ""
    results.append({**row.to_dict(), "valid_json": valid_json, "n_steps": n_steps, "support_level": support})

df_fixed = pd.DataFrame(results)
print(df_fixed["valid_json"].value_counts())

# Overwrite the CSV
df_fixed.to_csv(path, index=False, encoding="utf-8")
print("Saved.")

In [ ]:
import re, json, pandas as pd

def clean_output(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

def extract_actions(json_str):
    try:
        data = json.loads(json_str)
        return " ".join(f"{s.get('action','')} {s.get('detail','')}".strip() for s in data.get("steps", []))
    except:
        return ""

results = []
for _, row in df_fixed.iterrows():
    cleaned = clean_output(row["model_output"])
    try:
        parsed = json.loads(cleaned)
        valid_json = True
        n_steps = parsed.get("n_steps", len(parsed.get("steps", [])))
        support = parsed.get("support_level", "")
    except:
        valid_json = False
        n_steps = 0
        support = ""
    r = row.to_dict()
    r["model_output"] = cleaned  # overwrite with clean version
    r["valid_json"] = valid_json
    r["n_steps"] = n_steps
    r["support_level"] = support
    results.append(r)

df_fixed = pd.DataFrame(results)
df_fixed.to_csv(path, index=False, encoding="utf-8")
print(df_fixed["valid_json"].value_counts())

# Cell 8: Metrics evaluation for Zero-shot Phi-3 Mini

In [ ]:
!pip install rouge-score sacrebleu -q

import json, pandas as pd
from rouge_score import rouge_scorer
from sacrebleu.metrics import BLEU

def compute_metrics(df_results, model_name):
    def extract_actions(json_str):
        try:
            data = json.loads(json_str)
            return " ".join(
                f"{s.get('action','')} {s.get('detail','')}".strip()
                for s in data.get("steps", [])
            )
        except:
            return ""

    hyps, refs = [], []
    for _, row in df_results.iterrows():
        h = extract_actions(str(row["model_output"]))
        r = extract_actions(str(row["gold_output"]))
        if h and r:
            hyps.append(h)
            refs.append(r)

    scorer = rouge_scorer.RougeScorer(
        ["rouge1","rouge2","rougeL"], use_stemmer=True
    )
    r1 = r2 = rL = 0
    for h, r in zip(hyps, refs):
        s = scorer.score(r, h)
        r1 += s["rouge1"].fmeasure
        r2 += s["rouge2"].fmeasure
        rL += s["rougeL"].fmeasure
    n = len(hyps)
    bleu = BLEU(effective_order=True)
    bleu_score = bleu.corpus_score(hyps, [refs]).score

    print(f"\n── {model_name} ──────────────────────────")
    print(f"Valid pairs : {n}")
    print(f"ROUGE-1     : {r1/n:.4f}")
    print(f"ROUGE-2     : {r2/n:.4f}")
    print(f"ROUGE-L     : {rL/n:.4f}")
    print(f"BLEU-4      : {bleu_score:.4f}")
    return {"model": model_name, "valid_json": len(df_results[df_results['valid_json']==True]),
            "rouge1": r1/n, "rouge2": r2/n, "rougeL": rL/n, "bleu4": bleu_score}

RESULTS_DIR = "/content/drive/MyDrive/VisualStep/results"

all_results = []

for filename, label in [
    ("test_outputs_visualstep.csv",        "VisualStep (fine-tuned)"),
    ("test_outputs_phi3_zeroshot.csv",     "Zero-shot Phi-3 Mini"),
    ("test_outputs_qwen27b_zeroshot.csv",  "Zero-shot Qwen3.8 27B"),
    ("test_outputs_gemma2_2b_zeroshot.csv","Zero-shot Gemma 2 2B"),
]:
    import os
    path = os.path.join(RESULTS_DIR, filename)
    if os.path.exists(path):
        df = pd.read_csv(path)
        all_results.append(compute_metrics(df, label))
    else:
        print(f"\n⚠️  Not found: {filename}")

print("\n\n══ SUMMARY TABLE ══════════════════════════════")
df_summary = pd.DataFrame(all_results)
print(df_summary.to_string(index=False))

# Cell 9 — ACI Constraint Compliance

In [ ]:
import json, pandas as pd, spacy

nlp = spacy.load("en_core_web_sm")

def is_imperative(action_text):
    doc = nlp(action_text.strip())
    if len(doc) == 0:
        return False
    return doc[0].tag_ == "VB"

def check_aci(json_str):
    try:
        data = json.loads(str(json_str))
        steps = data.get("steps", [])
    except:
        return {"aci_valid": False, "c_ok": False, "a_ok": False, "i_ok": False, "n_steps": 0}

    n = len(steps)
    c_ok = (1 <= n <= 5)
    a_ok = all(len((s.get("action") or "").split()) <= 6 for s in steps)
    i_ok = all(is_imperative(s.get("action") or "") for s in steps)

    return {
        "aci_valid": c_ok and a_ok and i_ok,
        "c_ok": c_ok,
        "a_ok": a_ok,
        "i_ok": i_ok,
        "n_steps": n,
    }

RESULTS_DIR = "/content/drive/MyDrive/VisualStep/results"

for filename, label in [
    ("test_outputs_visualstep.csv",         "VisualStep (fine-tuned)"),
    ("test_outputs_phi3_zeroshot.csv",      "Zero-shot Phi-3 Mini"),
    ("test_outputs_qwen27b_zeroshot.csv",   "Zero-shot Qwen3.8 27B"),
    ("test_outputs_gemma2_2b_zeroshot.csv", "Zero-shot Gemma 2 2B"),
]:
    df = pd.read_csv(f"{RESULTS_DIR}/{filename}")
    aci = df["model_output"].apply(check_aci).apply(pd.Series)
    n = len(df)
    print(f"\n── {label} ──")
    print(f"  ACI fully valid : {aci['aci_valid'].sum()}/{n} ({aci['aci_valid'].mean()*100:.1f}%)")
    print(f"  C (≤5 steps)    : {aci['c_ok'].sum()}/{n} ({aci['c_ok'].mean()*100:.1f}%)")
    print(f"  A (≤6 words)    : {aci['a_ok'].sum()}/{n} ({aci['a_ok'].mean()*100:.1f}%)")
    print(f"  I (imperative)  : {aci['i_ok'].sum()}/{n} ({aci['i_ok'].mean()*100:.1f}%)")

# Cell 10 — Bootstrap Significance Test (BLEU-4)

In [ ]:
import json, numpy as np, pandas as pd
from sacrebleu.metrics import BLEU

RESULTS_DIR = "/content/drive/MyDrive/VisualStep/results"
N_BOOT = 1000
np.random.seed(42)

def extract_actions(json_str):
    try:
        data = json.loads(str(json_str))
        return " ".join(f"{s.get('action','')} {s.get('detail','')}".strip()
                        for s in data.get("steps", []))
    except:
        return ""

def bleu_score(hyps, refs):
    pairs = [(h, r) for h, r in zip(hyps, refs) if h and r]
    if not pairs:
        return 0.0
    h, r = zip(*pairs)
    return BLEU(effective_order=True).corpus_score(list(h), [list(r)]).score

def bootstrap_test(hyps_a, hyps_b, refs, n=N_BOOT):
    """Returns p-value: probability that B >= A under resampling."""
    idx = np.arange(len(refs))
    count = 0
    for _ in range(n):
        sample = np.random.choice(idx, size=len(idx), replace=True)
        sa = bleu_score([hyps_a[i] for i in sample], [refs[i] for i in sample])
        sb = bleu_score([hyps_b[i] for i in sample], [refs[i] for i in sample])
        if sb >= sa:
            count += 1
    return count / n  # p-value: how often baseline beats VisualStep

# Load VisualStep
vs = pd.read_csv(f"{RESULTS_DIR}/test_outputs_visualstep.csv")
refs = [extract_actions(r) for r in vs["gold_output"]]
hyps_vs = [extract_actions(r) for r in vs["model_output"]]

for filename, label in [
    ("test_outputs_phi3_zeroshot.csv",      "Zero-shot Phi-3 Mini"),
    ("test_outputs_qwen27b_zeroshot.csv",   "Zero-shot Qwen3.8 27B"),
    ("test_outputs_gemma2_2b_zeroshot.csv", "Zero-shot Gemma 2 2B"),
]:
    df = pd.read_csv(f"{RESULTS_DIR}/{filename}")
    hyps_b = [extract_actions(r) for r in df["model_output"]]
    p = bootstrap_test(hyps_vs, hyps_b, refs)
    sig = "✓ significant (p<0.05)" if p < 0.05 else "✗ NOT significant"
    print(f"{label}: p={p:.4f}  {sig}")